# Інференс

In [1]:
import os, re, glob, sys, time, math, subprocess, warnings
from pathlib import Path
from types import SimpleNamespace
import numpy as np, pandas as pd
import torch, torch.nn as nn, torchaudio, timm
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

try:
    import onnxruntime as ort
except ImportError:
    whl = glob.glob("/kaggle/input/**/onnxruntime-*.whl", recursive=True)
    if not whl: raise RuntimeError("Немає колеса onnxruntime у /kaggle/input — підключи датасет")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", whl[0]],
                   check=True)
    import onnxruntime as ort
print("onnxruntime", ort.__version__, "|", ort.get_available_providers())


class CFG:
    seed = 42; n_folds = 3
    sr = 32000; duration = 15; n_samples = sr * 15
    window_sec = 5; window_samples = sr * 5
    n_fft = 2048; hop_length = 512; n_mels = 128; fmin = 20; fmax = 16000
    batch_size = 12

    data_dir = "/kaggle/input/competitions/birdclef-2026"
    test_dir = "/kaggle/input/competitions/birdclef-2026/test_soundscapes"
    sample_sub = "/kaggle/input/competitions/birdclef-2026/sample_submission.csv"
    model_dir = "/kaggle/input/datasets/denyspushkaruks/rnn-weights-10"
    species_list_path = "/kaggle/input/datasets/denyspushkaruks/rnn-weights-10/species_list.txt"
    model_globs = ["kd_rnn2_fold_*_best.pth"]

    use_perch = True
    perch_backend = "onnx"
    perch_onnx = "/kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/perch_v2_no_dft.onnx"
    perch_path = "/kaggle/input/models"
    perch_threads = 4
    perch_w = 0.20
    blend = "geo"
    probe = False

    n_test_expected = 700

    PP_SLIDE_W = 0.85; PP_SLIDE_WIN = 5
    PP_FILE_W = 0.90; PP_GAUSS_SIGMA = 1.0
    PRIOR_LAMBDA = 0.4


dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", dev)

onnxruntime 1.24.4 | ['AzureExecutionProvider', 'CPUExecutionProvider']
Device: cpu


In [2]:
class MelSpec(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            c.sr, c.n_fft, hop_length=c.hop_length, n_mels=c.n_mels,
            f_min=c.fmin, f_max=c.fmax, power=2.0)
        self.db = torchaudio.transforms.AmplitudeToDB(top_db=80)
    def forward(self, x):
        m = self.db(self.mel(x))
        m = (m - m.mean((-2, -1), keepdim=True)) / (m.std((-2, -1), keepdim=True) + 1e-6)
        return m.unsqueeze(1)


class LSEPool(nn.Module):
    def __init__(self, r=5.0):
        super().__init__()
        self.r = nn.Parameter(torch.tensor(r))
    def forward(self, x):
        return torch.logsumexp(self.r * x, dim=1) / self.r


class SEDModel(nn.Module):
    def __init__(self, cfg, nc):
        super().__init__()
        self.backbone = timm.create_model(cfg.model_name, pretrained=False,
                                          in_chans=cfg.in_channels, num_classes=0, global_pool="")
        with torch.no_grad():
            o = self.backbone(torch.randn(1, cfg.in_channels, cfg.n_mels, 938))
            if o.dim() == 4: _, c, h, _ = o.shape; self.fd = c * h; self.ch = c
            else: self.fd = o.shape[-1]; self.ch = self.fd
        fp = getattr(cfg, "rnn_freq_pool", None)
        rnn_in = getattr(cfg, "rnn_in", None)
        if fp is None:
            fp = (rnn_in == self.ch) if rnn_in else False
        self.freq_pool = bool(fp) and cfg.use_rnn
        d = self.ch if self.freq_pool else self.fd
        self.rnn = None
        if cfg.use_rnn:
            self.rnn = nn.GRU(d, cfg.rnn_hidden, cfg.rnn_layers,
                              batch_first=True, bidirectional=True)
            d = cfg.rnn_hidden * 2
        self.fc = nn.Linear(d, nc)
        self.lse = LSEPool(r=5.0)
        self.drop = nn.Dropout(0.3)
        self.proj = nn.Linear(d, cfg.emb_dim) if cfg.emb_kd_w > 0 else None

    def forward(self, x):
        f = self.backbone(x)
        if f.dim() == 4:
            b, c, h, w = f.shape
            if self.freq_pool:
                f = f.mean(dim=2).permute(0, 2, 1)
            else:
                f = f.permute(0, 3, 1, 2).reshape(b, w, c * h)
        if self.rnn is not None:
            f, _ = self.rnn(f)
        f = self.drop(f)
        framewise = self.fc(f)
        return self.lse(framewise), framewise.max(dim=1).values


def load_students(cfg, nc):
    files = []
    for g in cfg.model_globs: files += sorted(glob.glob(os.path.join(cfg.model_dir, g)))
    files = list(dict.fromkeys(files))
    out = []
    for f in files:
        ck = torch.load(f, map_location=dev, weights_only=False)
        cc = ck.get("cfg", {})
        sd = ck["model_state_dict"]

        wih = sd.get("rnn.weight_ih_l0")
        mc = SimpleNamespace(
            model_name=cc.get("model_name", "tf_efficientnet_b0.ns_jft_in1k"),
            in_channels=cc.get("in_channels", 1), n_mels=cc.get("n_mels", 128),
            use_rnn=cc.get("use_rnn", wih is not None),
            rnn_hidden=cc.get("rnn_hidden", int(wih.shape[0]) // 3 if wih is not None else 256),
            rnn_layers=cc.get("rnn_layers", 1),
            rnn_freq_pool=cc.get("rnn_freq_pool", None),
            rnn_in=int(wih.shape[1]) if wih is not None else None,
            emb_kd_w=cc.get("emb_kd_w", 0.0), emb_dim=cc.get("emb_dim", 1536))
        m = SEDModel(mc, cc.get("num_classes", nc))
        m.load_state_dict(sd); m.to(dev).eval()
        fold = int(re.search(r"fold_(\d+)", Path(f).name).group(1))
        out.append({"name": Path(f).name, "model": m, "fold": fold, "score": ck.get("score")})
        print(f"  {Path(f).name}: fold={fold} rnn={m.rnn is not None} "
              f"freq_pool={m.freq_pool} score={ck.get('score')}")
    if not out: print("!!! Студентів не знайдено")
    return out

In [3]:
def _norm(s): return str(s).strip().lower().replace("_", " ")


def find_saved_model(root):
    if not root or not os.path.isdir(root): return None
    for dp, _, fn in os.walk(root):
        if "saved_model.pb" in fn: return dp
    return None


def perch_class_index(assets_root, n_logits, species, cfg):
    sp2i = {s: i for i, s in enumerate(species)}
    key2sp = {_norm(s): i for i, s in enumerate(species)}
    sci_of = {}
    tax_path = os.path.join(cfg.data_dir, "taxonomy.csv")
    if os.path.exists(tax_path):
        tax = pd.read_csv(tax_path)
        sci = next((c for c in tax.columns if "scientific" in c.lower()), None)
        if sci:
            for r in tax.itertuples():
                i = sp2i.get(getattr(r, "primary_label", None))
                if i is None: continue
                s = _norm(getattr(r, sci))
                key2sp.setdefault(s, i); sci_of[i] = s

    idx = np.full(len(species), -1, np.int64)
    sci_col, sci_hits = None, 0
    for f in glob.glob(os.path.join(assets_root, "**", "*.csv"), recursive=True):
        try: t = pd.read_csv(f)
        except Exception: continue
        if len(t) != n_logits: continue
        for c in t.columns:
            vals = t[c].map(_norm)
            for row, v in enumerate(vals):
                i = key2sp.get(v)
                if i is not None and idx[i] < 0: idx[i] = row
            hits = sum(1 for v in vals if v in sci_of.values())
            if hits > sci_hits: sci_col, sci_hits = vals.tolist(), hits
    if (idx >= 0).sum() == 0:
        raise RuntimeError(f"Не знайшов таблицю класів на {n_logits} рядків у {assets_root}. "
                           f"Підключи Kaggle Model google/bird-vocalization-classifier (perch_v2_cpu)")

    direct = int((idx >= 0).sum())

    proxy = {}
    if sci_col is not None:
        genus_rows = {}
        for row, v in enumerate(sci_col):
            g = v.split()[0] if v and " " in v else None
            if g: genus_rows.setdefault(g, []).append(row)
        for i in range(len(species)):
            if idx[i] >= 0: continue
            s = sci_of.get(i)
            if not s or " " not in s: continue
            rows = genus_rows.get(s.split()[0])
            if rows: proxy[i] = np.array(rows, np.int64)

    mask = np.zeros(len(species), np.float32)
    mask[idx >= 0] = 1.0
    for i in proxy: mask[i] = 1.0
    print(f"Perch: {n_logits} класів | прямий мапінг {direct}/{len(species)}, "
          f"+{len(proxy)} через genus-proxy → сигнал на {int(mask.sum())}/{len(species)}")
    return idx, mask, proxy


def load_perch(cfg, species):
    if cfg.perch_backend == "onnx":
        import onnxruntime as ort
        so = ort.SessionOptions()
        so.intra_op_num_threads = cfg.perch_threads
        avail = ort.get_available_providers()
        providers = [p for p in ("CUDAExecutionProvider", "CPUExecutionProvider") if p in avail]

        onnx_path = cfg.perch_onnx
        if os.path.isdir(onnx_path):
            found = glob.glob(os.path.join(onnx_path, "**", "*.onnx"), recursive=True)
            if not found: raise RuntimeError(f"У {onnx_path} немає жодного .onnx")
            onnx_path = sorted(found, key=lambda p: "no_dft" not in p)[0]
            print(f"perch_onnx вказував на теку → беру {os.path.basename(onnx_path)}")
        sess = ort.InferenceSession(onnx_path, so, providers=providers)

        inp = sess.get_inputs()[0]
        shape = list(inp.shape)
        if len(shape) != 2 or (isinstance(shape[-1], int) and shape[-1] != cfg.window_samples):
            raise RuntimeError(f"ONNX чекає вхід {shape}, а не (N, {cfg.window_samples}). "
                               f"Схоже, в експорт не потрапив фронтенд Perch.")

        names = [o.name for o in sess.get_outputs()]
        li = next((i for i, n in enumerate(names) if n.lower() in ("label", "logits")), None)
        probe = sess.run(None, {inp.name: np.zeros((1, cfg.window_samples), np.float32)})
        if li is None:
            li = max((i for i, o in enumerate(probe) if o.ndim == 2),
                     key=lambda i: probe[i].shape[-1])
        n_logits = probe[li].shape[-1]
        if n_logits < 1000:
            raise RuntimeError(f"Вихід '{names[li]}' має лише {n_logits} значень — це не логіти "
                               f"Perch (~14795). Перевір, що в експорті є 'label'.")
        run = lambda w: sess.run([names[li]], {inp.name: np.ascontiguousarray(w, np.float32)})[0]
        print(f"Perch: ONNX ({providers[0]}) | логіти з виходу '{names[li]}' ({n_logits})")
    else:
        import tensorflow as tf
        path = find_saved_model(cfg.perch_path)
        if path is None: raise RuntimeError("Perch не знайдено — підключи Kaggle Model")
        infer = tf.saved_model.load(path).signatures["serving_default"]
        in_key = list(infer.structured_input_signature[1].keys())[0]
        out = infer(**{in_key: tf.zeros([1, cfg.window_samples], tf.float32)})
        flat = {k: v for k, v in out.items() if len(v.shape) == 2}
        log_key = max(flat, key=lambda k: int(flat[k].shape[-1]))
        n_logits = int(flat[log_key].shape[-1])
        run = lambda w: infer(**{in_key: tf.convert_to_tensor(w)})[log_key].numpy()
        print("Perch: TensorFlow SavedModel")

    idx, mask, proxy = perch_class_index(cfg.perch_path, n_logits, species, cfg)
    direct = idx >= 0
    return SimpleNamespace(run=run, mask=mask, proxy=proxy,
                           keep=direct, src=idx[direct])

## Проби

In [4]:
@torch.no_grad()
def student_probs(models, mel_fn, waves):
    m = mel_fn(waves.to(dev))
    ps = []
    for mod in models:
        clip, fmax = mod(m)
        ps.append(torch.sigmoid(0.5 * clip + 0.5 * fmax).float().cpu().numpy())
    return np.mean(ps, 0)


def perch_probs(P, waves5, nc):
    pr = 1.0 / (1.0 + np.exp(-P.run(waves5)))
    p = np.zeros((len(pr), nc), np.float32)
    p[:, P.keep] = pr[:, P.src]
    for i, rows in P.proxy.items():
        p[:, i] = pr[:, rows].max(axis=1)
    return p


def check_onnx_parity(cfg, species, n=4):
    a = load_perch(SimpleNamespace(**{**vars(cfg.__class__), "perch_backend": "onnx"}), species)
    b = load_perch(SimpleNamespace(**{**vars(cfg.__class__), "perch_backend": "tf"}), species)
    w = np.random.RandomState(0).randn(n, cfg.window_samples).astype(np.float32) * 0.05
    la, lb = a.run(w), b.run(w)
    d = float(np.abs(la - lb).max())
    rank = float(np.mean([np.corrcoef(x, y)[0, 1] for x, y in zip(la, lb)]))
    print(f"ONNX vs TF: max|Δлогітів|={d:.4f} | кореляція={rank:.5f} "
          f"{'OK' if d < 0.1 and rank > 0.999 else '← РОЗБІЖНІСТЬ, не сабміть!'}")


def load_audio(fp, cfg, offset=None, dur=None):
    try:
        if offset is None:
            w, sr = torchaudio.load(fp)
        else:
            w, sr = torchaudio.load(fp, frame_offset=int(offset * cfg.sr),
                                    num_frames=int(dur * cfg.sr))
        if w.shape[0] > 1: w = w.mean(0, keepdim=True)
        w = w.squeeze(0)
        if sr != cfg.sr: w = torchaudio.functional.resample(w, sr, cfg.sr)
    except Exception:
        return None
    if dur is not None:
        need = int(dur * cfg.sr)
        if w.shape[0] < need: w = torch.nn.functional.pad(w, (0, need - w.shape[0]))
        w = w[:need]
    return w


def predict_file(fp, students, mel_fn, P, cfg, nc):
    audio = load_audio(fp, cfg, offset=0, dur=60)
    if audio is None: return None, None

    win = audio.unfold(0, cfg.n_samples, cfg.window_samples)
    sp_win = student_probs([m["model"] for m in students], mel_fn, win)
    slots = np.zeros((12, nc), np.float32); cnt = np.zeros((12, nc), np.float32)
    for t in range(min(len(win), 10)):
        slots[t:t + 3] += sp_win[t]; cnt[t:t + 3] += 1
    sp = slots / np.maximum(cnt, 1)

    pp = None
    if P is not None:
        w5 = audio.unfold(0, cfg.window_samples, cfg.window_samples).numpy()
        pp = perch_probs(P, w5, nc)
    return sp, pp


def blend(sp, pp, mask, w, mode="geo", eps=1e-6):
    if pp is None or w <= 0: return sp
    out = sp.copy(); m = mask > 0
    a = np.clip(sp[:, m], eps, 1.0); b = np.clip(pp[:, m], eps, 1.0)
    out[:, m] = a ** (1 - w) * b ** w if mode == "geo" else (1 - w) * a + w * b
    return out

# Хелпери

In [6]:
def pcmap(yt, yp, pad=5):
    aps = []
    for i in range(yt.shape[1]):
        t = np.concatenate([yt[:, i], np.ones(pad)])
        p = np.concatenate([yp[:, i], np.ones(pad)])
        try: aps.append(average_precision_score(t, p))
        except Exception: aps.append(0.)
    return float(np.mean(aps))


def _parse_sec(v):
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip(); pts = s.split(":")
    try:
        if len(pts) == 3: return int(pts[0]) * 3600 + int(pts[1]) * 60 + float(pts[2])
        if len(pts) == 2: return int(pts[0]) * 60 + float(pts[1])
        return float(s)
    except Exception: return 0.


def oof_rows(cfg, species):
    df = pd.read_csv(os.path.join(cfg.data_dir, "train.csv"))
    if "rating" in df.columns:
        df = df[(df["rating"] >= 2.0) | (df["rating"] == 0)].reset_index(drop=True)
    df["filepath"] = df["filename"].apply(lambda x: os.path.join(cfg.data_dir, "train_audio", x))
    df["start_sec"] = np.nan; df["source"] = "audio"

    tsl = os.path.join(cfg.data_dir, "train_soundscapes_labels.csv")
    tsd = os.path.join(cfg.data_dir, "train_soundscapes")
    ts = pd.read_csv(tsl)
    ts["filepath"] = ts["filename"].apply(lambda x: os.path.join(tsd, x))
    ts["start_sec"] = ts["start"].apply(_parse_sec); ts["source"] = "sc"
    cols = ["filepath", "primary_label", "start_sec", "source"]
    df = pd.concat([df[cols], ts[cols]], ignore_index=True)

    df["_s"] = df["primary_label"].apply(lambda x: x.split(";")[0].strip() if isinstance(x, str) else "unk")
    skf = StratifiedKFold(cfg.n_folds, shuffle=True, random_state=cfg.seed)
    df["fold"] = -1
    for fi, (_, vi) in enumerate(skf.split(df, df["_s"])): df.loc[vi, "fold"] = fi
    return df[df["source"] == "sc"].reset_index(drop=True)


def tune_perch_weight(cfg, students, mel_fn, P, species, max_rows=1500):
    nc = len(species); sp2i = {s: i for i, s in enumerate(species)}
    rows = oof_rows(cfg, species)
    if len(rows) > max_rows:
        rows = rows.sample(max_rows, random_state=cfg.seed).reset_index(drop=True)

    Y, PS, PP = [], [], []
    for fold in sorted(rows["fold"].unique()):
        fm = [m["model"] for m in students if m["fold"] == fold]
        if not fm: continue
        sub = rows[rows["fold"] == fold]
        for b in tqdm(range(0, len(sub), cfg.batch_size), desc=f"OOF fold {fold}", leave=False):
            chunk = sub.iloc[b:b + cfg.batch_size]
            waves = [load_audio(r.filepath, cfg, r.start_sec, cfg.duration) for r in chunk.itertuples()]
            ok = [i for i, w in enumerate(waves) if w is not None]
            if not ok: continue
            W = torch.stack([waves[i] for i in ok])
            PS.append(student_probs(fm, mel_fn, W))

            w5 = W.reshape(len(ok) * 3, cfg.window_samples).numpy()
            pw = perch_probs(P, w5, nc).reshape(len(ok), 3, nc)
            PP.append(pw.max(1))

            y = np.zeros((len(ok), nc), np.float32)
            for j, i in enumerate(ok):
                lab = chunk.iloc[i]["primary_label"]
                if isinstance(lab, str):
                    for p_ in lab.split(";"):
                        if p_.strip() in sp2i: y[j, sp2i[p_.strip()]] = 1.0
            Y.append(y)

    Y = np.concatenate(Y).astype(int); PS = np.concatenate(PS); PP = np.concatenate(PP)
    print(f"\nOOF рядків: {len(Y)}")
    print(f"  студент сам:  {pcmap(Y, PS):.4f}")
    print(f"  perch сам:    {pcmap(Y, blend(np.full_like(PS, 1e-6), PP, P.mask, 1.0, cfg.blend)):.4f}")

    best = (0.0, pcmap(Y, PS))
    for w in np.arange(0.05, 0.75, 0.05):
        s = pcmap(Y, blend(PS, PP, P.mask, float(w), cfg.blend))
        print(f"  w={w:.2f} → {s:.4f}" + ("  <-- best" if s > best[1] else ""))
        if s > best[1]: best = (float(w), s)
    print(f"\nОбрано perch_w={best[0]:.2f} (pcmAP {best[1]:.4f}, було {pcmap(Y, PS):.4f})")
    return best[0]

## Постпроц

In [7]:
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})")


def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m: return None, -1
    _, site, _, hms = m.groups()
    return site, int(hms[:2])


def build_prior_table(cfg, species_list):
    lbl2idx = {s: i for i, s in enumerate(species_list)}; nc = len(species_list)
    tsl = os.path.join(cfg.data_dir, "train_soundscapes_labels.csv")
    if not os.path.exists(tsl): return None
    df = pd.read_csv(tsl).drop_duplicates().reset_index(drop=True)
    sites, hours = zip(*[parse_fname(fn) for fn in df["filename"]])
    df["site"] = sites; df["hour"] = hours

    Y = np.zeros((len(df), nc), np.float32)
    for i, row in df.iterrows():
        for lbl in [t.strip() for t in str(row["primary_label"]).split(";") if t.strip()]:
            if lbl in lbl2idx: Y[i, lbl2idx[lbl]] = 1.0

    site_p = {s: Y[df["site"] == s].mean(0) for s in df["site"].dropna().unique()}
    hour_p = {h: Y[df["hour"] == h].mean(0) for h in df["hour"].unique() if h >= 0}
    sh_p = {k: Y[list(i)].mean(0) for k, i in df.groupby(["site", "hour"]).groups.items()
            if k[1] >= 0 and k[0] is not None}
    return {"global": Y.mean(0), "site": site_p, "hour": hour_p, "sh": sh_p}


def apply_priors(pred_mat, row_ids, priors, lam, eps=1e-4):
    if priors is None or lam <= 0: return pred_mat
    result = pred_mat.copy(); gp = priors["global"]
    for ri, rid in enumerate(row_ids):
        site, hour = parse_fname(rid.rsplit("_", 1)[0] + ".ogg")
        p = gp.copy()
        if hour in priors["hour"]: p = 0.5 * priors["hour"][hour] + 0.5 * p
        if site in priors["site"]: p = 0.6 * priors["site"][site] + 0.4 * p
        if (site, hour) in priors["sh"]: p = 0.7 * priors["sh"][(site, hour)] + 0.3 * p
        p = np.clip(p, eps, 1 - eps)
        pr = np.clip(result[ri], eps, 1 - eps)
        result[ri] = 1.0 / (1.0 + np.exp(-(np.log(pr / (1 - pr)) + lam * np.log(p / (1 - p)))))
    return result


def postprocess(pred_mat, row_ids, cfg):
    groups = {}
    for i, rid in enumerate(row_ids):
        groups.setdefault(rid.rsplit("_", 1)[0], []).append(i)
    for k in groups:
        groups[k].sort(key=lambda i: int(row_ids[i].rsplit("_", 1)[1]))

    p = pred_mat.copy()
    for idxs in groups.values():
        chunk = pred_mat[idxs]; T = len(idxs)
        for t in range(T):
            lo = max(0, t - cfg.PP_SLIDE_WIN // 2); hi = min(T, t + cfg.PP_SLIDE_WIN // 2 + 1)
            p[idxs[t]] = cfg.PP_SLIDE_W * chunk[t] + (1 - cfg.PP_SLIDE_W) * chunk[lo:hi].max(0)
    p2 = p.copy()
    for idxs in groups.values():
        chunk = p[idxs]
        p2[idxs] = cfg.PP_FILE_W * chunk + (1 - cfg.PP_FILE_W) * chunk.max(0, keepdims=True)
    p3 = p2.copy()
    for idxs in groups.values():
        if len(idxs) > 1:
            p3[idxs] = gaussian_filter1d(p2[idxs], sigma=cfg.PP_GAUSS_SIGMA, axis=0)
    return np.clip(p3, 0.0, 1.0)

In [8]:
def run(cfg=None, tune=True, output="submission.csv"):
    cfg = cfg or CFG(); t0 = time.time()
    species = [l.strip() for l in open(cfg.species_list_path) if l.strip()]
    nc = len(species); sp2i = {s: i for i, s in enumerate(species)}

    students = load_students(cfg, nc)
    mel_fn = MelSpec(cfg).to(dev)
    P = load_perch(cfg, species) if cfg.use_perch else None

    tfs = sorted(sum([glob.glob(os.path.join(cfg.test_dir, e)) for e in ("*.ogg", "*.wav", "*.flac")], []))
    print(f"Test: {len(tfs)} файлів")

    sub_df = pd.read_csv(cfg.sample_sub)
    if not tfs or not students:
        print("Немає тесту/моделей → sample submission"); sub_df.to_csv(output, index=False); return

    if cfg.probe:
        probe_speed(tfs, students, mel_fn, P, cfg, nc, n=min(5, len(tfs)),
                    n_test_expected=cfg.n_test_expected)
    if tune and P is not None:
        cfg.perch_w = tune_perch_weight(cfg, students, mel_fn, P, species)
    print(f"\nperch_w={cfg.perch_w:.2f} | blend={cfg.blend}")

    preds = {}
    for fp in tqdm(tfs, desc="Inference"):
        sp, pp = predict_file(fp, students, mel_fn, P, cfg, nc)
        if sp is None: continue
        pr = blend(sp, pp, P.mask, cfg.perch_w, cfg.blend) if P is not None else sp
        stem = Path(fp).stem
        for slot in range(12): preds[f"{stem}_{(slot + 1) * 5}"] = pr[slot]

    sub_sp = [c for c in sub_df.columns if c != "row_id"]
    row_ids = sub_df["row_id"].tolist()
    mat = np.zeros((len(sub_df), len(sub_sp)), np.float32); matched = 0
    for ri, rid in enumerate(row_ids):
        if rid in preds:
            pv = preds[rid]; matched += 1
            for ci, s in enumerate(sub_sp):
                if s in sp2i: mat[ri, ci] = pv[sp2i[s]]

    mat = apply_priors(mat, row_ids, build_prior_table(cfg, sub_sp), cfg.PRIOR_LAMBDA)
    mat = postprocess(mat, row_ids, cfg)

    sub = pd.DataFrame(mat, columns=sub_sp)
    sub.insert(0, "row_id", sub_df["row_id"].values)
    sub.to_csv(output, index=False)
    print(f"\nDone: {sub.shape} | matched {matched}/{len(sub_df)} | {(time.time() - t0) / 60:.1f}min")


run(CFG(), tune=False)

  kd_rnn2_fold_0_best.pth: fold=0 rnn=True freq_pool=True score=0.8207918565758135
  kd_rnn2_fold_1_best.pth: fold=1 rnn=True freq_pool=True score=0.8185291980818022
  kd_rnn2_fold_2_best.pth: fold=2 rnn=True freq_pool=True score=0.8152561656848563
Perch: ONNX (CPUExecutionProvider) | логіти з виходу 'label' (14795)
Perch: 14795 класів | прямий мапінг 203/234, +6 через genus-proxy → сигнал на 209/234
Test: 0 файлів
Немає тесту/моделей → sample submission
